<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/preprocess_for_tfidf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import string
import pandas as pd

# Base directory where the 10 TSV files are stored
BASE_DIR = "/content/drive/MyDrive/world-inflation/result/tsv/"

# Target subreddits
SUBREDDITS = ["food", "cars", "travel", "real_estate", "frugal"]


In [ ]:
def load_and_merge_subreddit(subreddit: str, base_dir: str) -> pd.DataFrame:
    """
    Load submission and comment TSVs for a subreddit, merge them,
    add a 'type' column, convert date, deduplicate, and filter by inflation label.
    """
    # Build file paths
    sub_file = os.path.join(base_dir, f"rerun_{subreddit}_submissions_results.tsv")
    com_file = os.path.join(base_dir, f"rerun_{subreddit}_comments_results.tsv")

    # Read TSV files
    sub_df = pd.read_csv(sub_file, sep="\t")
    com_df = pd.read_csv(com_file, sep="\t")

    # Add type column
    sub_df["type"] = "submission"
    com_df["type"] = "comment"

    # Concatenate
    df = pd.concat([sub_df, com_df], ignore_index=True)

    # Convert created_date to datetime if present
    if "created_date" in df.columns:
        df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")

    # Ensure inflation is numeric
    if "inflation" in df.columns:
        df["inflation"] = pd.to_numeric(df["inflation"], errors="coerce")
    else:
        raise ValueError("Column 'inflation' is missing in the TSV files.")

    # Filter for inflation == 2 (inflation class only)
    df = df[df["inflation"] == 2].copy()

    # Drop duplicate rows based on Reddit id (keep the first occurrence)
    if "id" not in df.columns:
        raise ValueError("Column 'id' is missing in the TSV files.")
    before = len(df)
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)
    after = len(df)

    print(f"{subreddit}: {before} rows after filtering by inflation==2, "
          f"{after} rows after dedup by id.")

    return df


In [ ]:
subreddit_dfs = {}

for sub in SUBREDDITS:
    df_sub = load_and_merge_subreddit(sub, BASE_DIR)
    subreddit_dfs[sub] = df_sub

# Quick sanity check
for sub, df_sub in subreddit_dfs.items():
    print(sub, df_sub.shape)


In [ ]:
def clean_text(text: str) -> str:
    """
    Basic text cleaning for TF-IDF input:
    - Lowercase
    - Remove URLs
    - Remove Reddit mentions (r/subreddit, u/user)
    - Remove bracketed tokens like [deleted]
    - Remove digits
    - Remove punctuation
    - Collapse multiple spaces
    """
    if pd.isna(text):
        return ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove Reddit-style mentions (r/xxx, u/xxx)
    text = re.sub(r"(r/|u/)[a-z0-9_]+", " ", text)

    # Remove bracketed tokens like [deleted], [removed]
    text = re.sub(r"\[.*?\]", " ", text)

    # Remove HTML entities (roughly)
    text = re.sub(r"&\w+;", " ", text)

    # Remove digits
    text = re.sub(r"\d+", " ", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Collapse multiple spaces and strip
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
for sub, df_sub in subreddit_dfs.items():
    if "body" not in df_sub.columns:
        raise ValueError(f"'body' column not found for subreddit {sub}.")

    # Apply cleaning function
    df_sub["body_clean"] = df_sub["body"].astype(str).apply(clean_text)

    # Optional: quick preview
    print(f"\n{sub}: sample cleaned text")
    print(df_sub[["body", "body_clean"]].head(3))

    # Store back
    subreddit_dfs[sub] = df_sub


In [ ]:
for sub, df_sub in subreddit_dfs.items():
    out_path = os.path.join(BASE_DIR, f"rerun-tfidf_input_{sub}.tsv")
    df_sub.to_csv(out_path, sep="\t", index=False)
    print(f"Saved: {out_path} (rows={len(df_sub)})")
